In [1]:
from pathlib import Path
from collections import Counter
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from rank_bm25 import BM25Okapi

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

In [2]:
cwd = Path.cwd()
BASE_DIR = cwd.parent if cwd.name == "notebooks" else cwd

PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"

print(BASE_DIR)

c:\Users\rlagu\Desktop\news-recommender-project


In [3]:
train_features = pd.read_csv(PROCESSED_DIR / "train_airs_lite_features.csv")
dev_features = pd.read_csv(PROCESSED_DIR / "dev_airs_lite_features.csv")
news_all = pd.read_csv(PROCESSED_DIR / "news_all.csv")

train_features["history"] = train_features["history"].fillna("")
dev_features["history"] = dev_features["history"].fillna("")
news_all["text"] = news_all["text"].fillna("")

print("train_features:", train_features.shape)
print("dev_features:", dev_features.shape)
print("news_all:", news_all.shape)

train_features: (187715, 10)
dev_features: (187009, 10)
news_all: (65238, 6)


In [4]:
def tokenize(text):
    text = str(text).lower()
    tokens = re.findall(r"[a-z0-9]+", text)
    return tokens

In [5]:
news_all["tokens"] = news_all["text"].apply(tokenize)

news_id_to_idx = {
    news_id: idx
    for idx, news_id in enumerate(news_all["news_id"])
}

news_id_to_tokens = {
    row["news_id"]: row["tokens"]
    for _, row in news_all.iterrows()
}

news_all[["news_id", "text", "tokens"]].head()

,news_id,text,tokens
0,N55528,"The Brands Queen Elizabeth, Prince Charles, an...","[the, brands, queen, elizabeth, prince, charle..."
1,N19639,50 Worst Habits For Belly Fat These seemingly ...,"[50, worst, habits, for, belly, fat, these, se..."
2,N61837,The Cost of Trump's Aid Freeze in the Trenches...,"[the, cost, of, trump, s, aid, freeze, in, the..."
3,N53526,I Was An NBA Wife. Here's How It Affected My M...,"[i, was, an, nba, wife, here, s, how, it, affe..."
4,N38324,"How to Get Rid of Skin Tags, According to a De...","[how, to, get, rid, of, skin, tags, according,..."


In [6]:
tokenized_corpus = news_all["tokens"].tolist()

bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index built:", len(tokenized_corpus))

BM25 index built: 65238


In [7]:
def get_history_news_ids(history, max_history=30):
    if pd.isna(history) or history == "":
        return []
    return str(history).split()[-max_history:]


def build_user_query_tokens(history, max_history=30, max_query_tokens=300):
    history_ids = get_history_news_ids(history, max_history=max_history)
    
    query_tokens = []
    
    for news_id in history_ids:
        query_tokens.extend(news_id_to_tokens.get(news_id, []))
    
    # 너무 길면 최근/앞쪽 토큰 일부만 사용
    if len(query_tokens) > max_query_tokens:
        query_tokens = query_tokens[-max_query_tokens:]
    
    return query_tokens

In [8]:
def add_bm25_scores(features_df, name="features", max_history=30, max_query_tokens=300):
    result_groups = []

    for impression_id, group in tqdm(features_df.groupby("impression_id"), desc=f"BM25 scoring: {name}"):
        group = group.copy()

        history = group["history"].iloc[0]
        query_tokens = build_user_query_tokens(
            history,
            max_history=max_history,
            max_query_tokens=max_query_tokens
        )

        candidate_indices = [
            news_id_to_idx.get(news_id, None)
            for news_id in group["candidate_news"]
        ]

        bm25_scores = []

        if len(query_tokens) == 0:
            bm25_scores = [0.0] * len(group)
        else:
            valid_positions = []
            valid_doc_indices = []

            for pos, idx in enumerate(candidate_indices):
                if idx is not None:
                    valid_positions.append(pos)
                    valid_doc_indices.append(idx)

            scores = [0.0] * len(group)

            if len(valid_doc_indices) > 0:
                # rank_bm25는 일부 버전에서 get_batch_scores 지원
                if hasattr(bm25, "get_batch_scores"):
                    batch_scores = bm25.get_batch_scores(query_tokens, valid_doc_indices)
                else:
                    all_scores = bm25.get_scores(query_tokens)
                    batch_scores = [all_scores[idx] for idx in valid_doc_indices]

                for pos, score in zip(valid_positions, batch_scores):
                    scores[pos] = float(score)

            bm25_scores = scores

        group["bm25_score"] = bm25_scores
        result_groups.append(group)

    return pd.concat(result_groups, ignore_index=True)

In [9]:
train_bm25_features = add_bm25_scores(train_features, name="train")
dev_bm25_features = add_bm25_scores(dev_features, name="dev")

print(train_bm25_features.shape)
print(dev_bm25_features.shape)

dev_bm25_features[["impression_id", "candidate_news", "label", "tfidf_score", "bm25_score"]].head()

BM25 scoring: train:   0%|          | 0/5000 [00:00<?, ?it/s]

BM25 scoring: dev:   0%|          | 0/5000 [00:00<?, ?it/s]

(187715, 11)
(187009, 11)


,impression_id,candidate_news,label,tfidf_score,bm25_score
0,1,N28682,0,0.003380,80.526186
1,1,N48740,0,0.008121,130.156986
2,1,N31958,1,0.005173,145.031690
3,1,N34130,0,0.001394,42.534154
4,1,N6916,0,0.003035,130.951743


In [10]:
train_bm25_features.to_csv(
    PROCESSED_DIR / "train_airs_lite_bm25_features.csv",
    index=False,
    encoding="utf-8-sig"
)

dev_bm25_features.to_csv(
    PROCESSED_DIR / "dev_airs_lite_bm25_features.csv",
    index=False,
    encoding="utf-8-sig"
)

print("saved BM25 features")

saved BM25 features


In [11]:
def mrr_score(labels, scores):
    labels = np.array(labels)
    scores = np.array(scores)

    order = np.argsort(scores)[::-1]
    sorted_labels = labels[order]

    positive_indices = np.where(sorted_labels == 1)[0]

    if len(positive_indices) == 0:
        return np.nan

    return 1.0 / (positive_indices[0] + 1)


def dcg_at_k(labels, scores, k):
    labels = np.array(labels)
    scores = np.array(scores)

    order = np.argsort(scores)[::-1]
    sorted_labels = labels[order][:k]

    gains = sorted_labels
    discounts = np.log2(np.arange(2, len(sorted_labels) + 2))

    return np.sum(gains / discounts)


def ndcg_at_k(labels, scores, k):
    dcg = dcg_at_k(labels, scores, k)

    ideal_scores = np.array(labels)
    idcg = dcg_at_k(labels, ideal_scores, k)

    if idcg == 0:
        return np.nan

    return dcg / idcg


def evaluate_by_impression(scored_df):
    metrics = []

    for impression_id, group in scored_df.groupby("impression_id"):
        labels = group["label"].values
        scores = group["score"].values

        if len(np.unique(labels)) < 2:
            auc = np.nan
        else:
            auc = roc_auc_score(labels, scores)

        metrics.append({
            "impression_id": impression_id,
            "AUC": auc,
            "MRR": mrr_score(labels, scores),
            "nDCG@5": ndcg_at_k(labels, scores, k=5),
            "nDCG@10": ndcg_at_k(labels, scores, k=10)
        })

    metrics_df = pd.DataFrame(metrics)
    result = metrics_df[["AUC", "MRR", "nDCG@5", "nDCG@10"]].mean(skipna=True)

    return result, metrics_df

In [12]:
def train_and_evaluate(selected_features, model_name):
    X_train = train_bm25_features[selected_features].fillna(0)
    y_train = train_bm25_features["label"].astype(int)

    X_dev = dev_bm25_features[selected_features].fillna(0)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)

    scored_dev = dev_bm25_features.copy()
    scored_dev["score"] = model.predict_proba(X_dev)[:, 1]

    result, metrics_df = evaluate_by_impression(scored_dev)

    coef_table = pd.DataFrame({
        "feature": selected_features,
        "coef": model.named_steps["lr"].coef_[0],
        "model": model_name
    }).sort_values("coef", ascending=False)

    return {
        "model": model_name,
        "AUC": result["AUC"],
        "MRR": result["MRR"],
        "nDCG@5": result["nDCG@5"],
        "nDCG@10": result["nDCG@10"]
    }, coef_table, metrics_df

In [13]:
model_settings = {
    "TF-IDF baseline": ["tfidf_score"],
    "BM25 baseline": ["bm25_score"],

    "AiRS-lite revised": [
        "npmi_score",
        "category_pref",
        "subcategory_pref",
        "tfidf_score"
    ],

    "AiRS-lite BM25": [
        "npmi_score",
        "category_pref",
        "subcategory_pref",
        "bm25_score"
    ],

    "AiRS-lite hybrid": [
        "npmi_score",
        "category_pref",
        "subcategory_pref",
        "tfidf_score",
        "bm25_score"
    ],

    "Category + BM25": [
        "category_pref",
        "subcategory_pref",
        "bm25_score"
    ],

    "Category + TF-IDF": [
        "category_pref",
        "subcategory_pref",
        "tfidf_score"
    ]
}

bm25_results = []
coef_tables = []

for model_name, features in model_settings.items():
    print("running:", model_name, features)

    result_row, coef_table, metrics_df = train_and_evaluate(features, model_name)

    bm25_results.append(result_row)
    coef_tables.append(coef_table)

bm25_results_df = pd.DataFrame(bm25_results)
bm25_coef_df = pd.concat(coef_tables, ignore_index=True)

bm25_results_df.sort_values("AUC", ascending=False)

running: TF-IDF baseline ['tfidf_score']
running: BM25 baseline ['bm25_score']
running: AiRS-lite revised ['npmi_score', 'category_pref', 'subcategory_pref', 'tfidf_score']
running: AiRS-lite BM25 ['npmi_score', 'category_pref', 'subcategory_pref', 'bm25_score']
running: AiRS-lite hybrid ['npmi_score', 'category_pref', 'subcategory_pref', 'tfidf_score', 'bm25_score']
running: Category + BM25 ['category_pref', 'subcategory_pref', 'bm25_score']
running: Category + TF-IDF ['category_pref', 'subcategory_pref', 'tfidf_score']


,model,AUC,MRR,nDCG@5,nDCG@10
6,Category + TF-IDF,0.631230,0.358192,0.340984,0.399949
5,Category + BM25,0.625578,0.349661,0.332238,0.390924
2,AiRS-lite revised,0.624651,0.331842,0.321276,0.380322
4,AiRS-lite hybrid,0.611041,0.326695,0.308556,0.369793
3,AiRS-lite BM25,0.606538,0.322458,0.304822,0.366687
0,TF-IDF baseline,0.577705,0.322725,0.297889,0.361070
1,BM25 baseline,0.530019,0.289190,0.262389,0.325452


In [14]:
bm25_results_df.to_csv(
    OUTPUT_DIR / "bm25_experiment_results.csv",
    index=False,
    encoding="utf-8-sig"
)

bm25_coef_df.to_csv(
    OUTPUT_DIR / "bm25_experiment_coef.csv",
    index=False,
    encoding="utf-8-sig"
)

print("saved!")

saved!
